# Inference with SK2Decompile

## Cấu hình `/kaggle/tmp` và kiểm tra đường dẫn Dataset

In [1]:
import os

# Bắt buộc trỏ cache HuggingFace sang /kaggle/tmp để không tràn 20GB của /kaggle/working
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf_cache"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

os.makedirs("/kaggle/tmp/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)

REPOS_DIR = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos"
SUBSET_PATH = "/kaggle/input/datasets/wood1enqxyn/llm4binary-subset/decompile-eval-subset-1000.jsonl"

!nvidia-smi
!ls -lh {SUBSET_PATH}

Fri Nov 20 07:45:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cài đặt thư viện

In [2]:
!pip install -q vllm transformers tree-sitter==0.24.0 tree-sitter-c==0.23.4 tqdm
!apt-get update -qq && apt-get install -y -qq clang-format

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is i

## Kiểm tra dữ liệu & tìm script mẫu trong repo

Trước khi chạy suy luận hàng loạt (mất nhiều giờ GPU), cần kiểm tra đúng tên cột trong `decompile-eval-subset-1000.jsonl` và xem mẫu prompt gốc trong repo **sk2decompile**.

In [3]:
import json

# 1. Xem 1 dòng đầu tiên của tập dữ liệu để biết chính xác tên key (vd: 'c_func', 'pseudo', 'opt'...)
with open(SUBSET_PATH, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())
print("Các key trong mỗi mẫu:", list(first_line.keys()))
print("Ví dụ opt:", first_line.get("opt"))

# 2. Tìm các file script suy luận có sẵn trong repo SK2Decompile để đối chiếu prompt format
!find {REPOS_DIR} -maxdepth 4 -name "*.py" | grep -iE "sk2|infer|decompile"

Các key trong mỗi mẫu: ['index', 'func_name', 'func_dep', 'func', 'test', 'opt', 'language', 'asm', 'ida_asm', 'ida_pseudo', 'ghidra_asm', 'ghidra_pseudo', 'pseudo', 'ida_pseudo_norm']
Ví dụ opt: O0
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/deepmath_103k.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/hh-rlhf-helpful-base.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/llava_instruct_mix.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/lm-human-preferences-descriptiveness.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/lm-human-preferences-sentiment.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/AutoDecompilers/examples/datasets/math_shepherd.py
/kaggle/input/datasets/wood1enqxyn/sk2decompile-autod

## Chạy Pha 1 – Structure Recovery (Skeleton)

* Tạo file `/kaggle/tmp/run_phase1.py` để load model `LLM4Binary/sk2decompile-struct-6.7b` (hoặc tên repo chính xác trên HuggingFace).
* Chạy theo batch để tận dụng tối đa tốc độ của vllm, ghi log thời gian/VRAM, lưu checkpoint `sk2_phase1_ir.jsonl`
* Xóa cache model Pha 1 trong `/kaggle/tmp` sau khi xong để chừa dung lượng đĩa cho Pha 2.

In [4]:
%%writefile /kaggle/tmp/run_phase1.py
import os
import json
import time
import subprocess
from vllm import LLM, SamplingParams

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

SUBSET_PATH = "/kaggle/input/datasets/wood1enqxyn/llm4binary-subset/decompile-eval-subset-1000.jsonl"
OUT_PHASE1 = "/kaggle/working/outputs/sk2_phase1_ir.jsonl"
LOG_FILE = "/kaggle/working/logs/sk2_inference_time.log"

def main():
    os.makedirs("/kaggle/working/outputs", exist_ok=True)
    os.makedirs("/kaggle/working/logs", exist_ok=True)

    # 1. Đọc dữ liệu đầu vào
    samples = []
    with open(SUBSET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))

    assert len(samples) > 0, "LỖI: File dataset rỗng!"
    print(f"[Pha 1] Tổng số mẫu: {len(samples)}")

    # 2. Chuẩn bị prompt (đối chiếu lại template ở Cell 3 trước khi chạy)
    prompts = []
    for s in samples:
        pseudo_code = s.get("ida_pseudo") or s.get("normalized_pseudo") or s.get("pseudo") or ""
        prompt = f"# This is the assembly/pseudo code:\n{pseudo_code}\n# What is the source code?\n"
        prompts.append(prompt)

    # 3. Load model Pha 1
    t_load_start = time.time()
    llm = LLM(
        model="LLM4Binary/sk2decompile-struct-6.7b",
        dtype="float16",
        tensor_parallel_size=2,
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        trust_remote_code=True,
    )
    t_load = time.time() - t_load_start

    # 4. Cắt ngắn các prompt quá dài (> 2048 tokens) để tránh crash batch
    tokenizer = llm.get_tokenizer()
    max_prompt_tokens = 2048
    safe_prompts = []
    truncated_count = 0
    for p in prompts:
        ids = tokenizer.encode(p, add_special_tokens=False)
        if len(ids) > max_prompt_tokens:
            p = tokenizer.decode(ids[:max_prompt_tokens], skip_special_tokens=True)
            truncated_count += 1
        safe_prompts.append(p)
    print(f"[Pha 1] Số mẫu bị cắt ngắn (> {max_prompt_tokens} tokens): {truncated_count}")

    sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)

    # 5. Chạy suy luận & đo thời gian
    t_infer_start = time.time()
    outputs = llm.generate(safe_prompts, sampling_params)
    t_infer = time.time() - t_infer_start

    vram_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"]
    ).decode("utf-8").strip()

    # 6. Lưu kết quả
    total_gen_tokens = 0
    with open(OUT_PHASE1, "w", encoding="utf-8") as f_out:
        for sample, out in zip(samples, outputs):
            gen_text = out.outputs[0].text
            total_gen_tokens += len(out.outputs[0].token_ids)
            sample["phase1_ir"] = gen_text
            f_out.write(json.dumps(sample, ensure_ascii=False) + "\n")

    # 7. Ghi log
    tps = total_gen_tokens / t_infer if t_infer > 0 else 0
    sec_per_func = t_infer / len(samples)
    log_msg = (
        f"=== PHA 1: STRUCTURE RECOVERY (sk2decompile-struct-6.7b) ===\n"
        f"Load model time: {t_load:.2f}s\n"
        f"Inference time: {t_infer:.2f}s ({sec_per_func:.3f} s/hàm)\n"
        f"Total generated tokens: {total_gen_tokens} | Speed: {tps:.2f} tokens/s\n"
        f"GPU VRAM snapshot:\n{vram_info}\n\n"
    )
    with open(LOG_FILE, "w", encoding="utf-8") as f_log:
        f_log.write(log_msg)

    print(log_msg)

if __name__ == "__main__":
    main()

Writing /kaggle/tmp/run_phase1.py


### Chạy script Pha 1 và dọn sạch cache model Pha 1 sau khi hoàn tất:

In [5]:
!python /kaggle/tmp/run_phase1.py

# Xóa cache model Pha 1 để giải phóng ~13.5GB dung lượng /kaggle/tmp cho Pha 2
!rm -rf /kaggle/tmp/hf_cache/hub/models--LLM4Binary--sk2decompile-struct-6.7b
!df -h /kaggle/tmp /kaggle/working

W1120 07:50:20.473000 1026 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1120 07:50:20.474000 1026 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1120 07:50:20.475000 1026 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
[Pha 1] Tổng số mẫu: 880
(MainProcess pid=1026) INFO 11-20 07:50:46 [startup.py:96] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'LLM4Binary/sk2decompile-struct-6.7b'}
config.json: 100%|█████████████████████████████| 778/778 [00:00<00:00, 3.09MB/s]
(MainProcess pid=1026) INFO 11-20 07:51:03 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=1026) INFO 11-20 07:51:03 [model.py:2557] Downcasting torch.float32 to torch.float16.
(MainProcess pid=1026) INFO 11-20 07

## Chạy Pha 2 – Identifier Naming (Skin)

In [6]:
%%writefile /kaggle/tmp/run_phase2.py
import os
import json
import time
import subprocess
from vllm import LLM, SamplingParams

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

IN_PHASE1 = "/kaggle/working/outputs/sk2_phase1_ir.jsonl"
OUT_PHASE2 = "/kaggle/working/outputs/sk2_recovered_code_c.jsonl"
LOG_FILE = "/kaggle/working/logs/sk2_inference_time.log"

def main():
    os.makedirs("/kaggle/working/outputs", exist_ok=True)
    os.makedirs("/kaggle/working/logs", exist_ok=True)

    # 1. Đọc kết quả IR từ Pha 1
    samples = []
    with open(IN_PHASE1, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))

    assert len(samples) > 0, "LỖI: File kết quả Pha 1 rỗng!"
    print(f"[Pha 2] Tổng số mẫu nhận từ Pha 1: {len(samples)}")

    # 2. Chuẩn bị prompt (đối chiếu lại template ở Cell 3 trước khi chạy)
    prompts = []
    for s in samples:
        ir_code = s.get("phase1_ir", "")
        prompt = f"# This is the intermediate representation:\n{ir_code}\n# Rename identifiers to make it readable:\n"
        prompts.append(prompt)

    # 3. Load model Pha 2
    t_load_start = time.time()
    llm = LLM(
        model="LLM4Binary/sk2decompile-ident-6.7b",
        dtype="float16",
        tensor_parallel_size=2,
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        trust_remote_code=True,
    )
    t_load = time.time() - t_load_start

    # 4. Cắt ngắn các prompt quá dài (> 2048 tokens) để tránh crash batch
    tokenizer = llm.get_tokenizer()
    max_prompt_tokens = 2048
    safe_prompts = []
    truncated_count = 0
    for p in prompts:
        ids = tokenizer.encode(p, add_special_tokens=False)
        if len(ids) > max_prompt_tokens:
            p = tokenizer.decode(ids[:max_prompt_tokens], skip_special_tokens=True)
            truncated_count += 1
        safe_prompts.append(p)
    print(f"[Pha 2] Số mẫu bị cắt ngắn (> {max_prompt_tokens} tokens): {truncated_count}")

    sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)

    # 5. Chạy suy luận Pha 2
    t_infer_start = time.time()
    outputs = llm.generate(safe_prompts, sampling_params)
    t_infer = time.time() - t_infer_start

    vram_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"]
    ).decode("utf-8").strip()

    # 6. Lưu kết quả code C hoàn chỉnh
    total_gen_tokens = 0
    with open(OUT_PHASE2, "w", encoding="utf-8") as f_out:
        for sample, out in zip(samples, outputs):
            gen_text = out.outputs[0].text
            total_gen_tokens += len(out.outputs[0].token_ids)
            sample["recovered_code"] = gen_text
            f_out.write(json.dumps(sample, ensure_ascii=False) + "\n")

    # 7. Ghi nối tiếp vào file log hiệu năng
    tps = total_gen_tokens / t_infer if t_infer > 0 else 0
    sec_per_func = t_infer / len(samples)
    log_msg = (
        f"=== PHA 2: IDENTIFIER NAMING (sk2decompile-ident-6.7b) ===\n"
        f"Load model time: {t_load:.2f}s\n"
        f"Inference time: {t_infer:.2f}s ({sec_per_func:.3f} s/hàm)\n"
        f"Total generated tokens: {total_gen_tokens} | Speed: {tps:.2f} tokens/s\n"
        f"GPU VRAM snapshot:\n{vram_info}\n"
    )
    with open(LOG_FILE, "a", encoding="utf-8") as f_log:
        f_log.write(log_msg)

    print(log_msg)

if __name__ == "__main__":
    main()

Writing /kaggle/tmp/run_phase2.py


In [7]:
!python /kaggle/tmp/run_phase2.py

W1120 09:07:09.095000 1864 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1120 09:07:09.096000 1864 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1120 09:07:09.096000 1864 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
[Pha 2] Tổng số mẫu nhận từ Pha 1: 880
(MainProcess pid=1864) INFO 11-20 09:07:23 [startup.py:96] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'LLM4Binary/sk2decompile-ident-6.7b'}
config.json: 100%|█████████████████████████████| 754/754 [00:00<00:00, 2.62MB/s]
(MainProcess pid=1864) INFO 11-20 09:07:24 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=1864) INFO 11-20 09:07:24 [model.py:2557] Downcasting torch.float32 to torch.float16.
(MainProcess pid=1864) 

## Kiểm tra lại file đầu ra trước khi tắt GPU

In [8]:
!ls -lh /kaggle/working/outputs/
!ls -lh /kaggle/working/logs/
!cat /kaggle/working/logs/sk2_inference_time.log
!head -n 1 /kaggle/working/outputs/sk2_recovered_code_c.jsonl

total 21M
-rw-r--r-- 1 root root 10M Nov 20 09:06 sk2_phase1_ir.jsonl
-rw-r--r-- 1 root root 11M Nov 20 10:18 sk2_recovered_code_c.jsonl
total 4.0K
-rw-r--r-- 1 root root 495 Nov 20 10:18 sk2_inference_time.log
=== PHA 1: STRUCTURE RECOVERY (sk2decompile-struct-6.7b) ===
Load model time: 413.16s
Inference time: 4153.99s (4.720 s/hàm)
Total generated tokens: 312464 | Speed: 75.22 tokens/s
GPU VRAM snapshot:
0, 13007 MiB, 15360 MiB
1, 13007 MiB, 15360 MiB

=== PHA 2: IDENTIFIER NAMING (sk2decompile-ident-6.7b) ===
Load model time: 386.06s
Inference time: 3904.69s (4.437 s/hàm)
Total generated tokens: 314489 | Speed: 80.54 tokens/s
GPU VRAM snapshot:
0, 12999 MiB, 15360 MiB
1, 12999 MiB, 15360 MiB
{"index": 0, "func_name": "mi_cmp_dynamic_unique", "func_dep": "eloqsql/storage/myisam/mi_dynrec.c", "func": "int _mi_cmp_dynamic_unique(MI_INFO *info, MI_UNIQUEDEF *def,\n\t\t\t   const uchar *record, my_off_t pos)\n{\n  uchar *rec_buff,*old_record;\n  int error;\n  DBUG_ENTER(\"_mi_cmp_dynamic